# Inferencia REST contra `model_serving`

Este notebook consume la API FastAPI definida en `../model_serving/main.py` usando la librería `requests`. Se cubren los dos endpoints:

1. **`POST /upload_model/`** — sube los tres `.pkl` (preprocesador, filtro y calibrador Venn-Abers) que produjo el notebook `12_complete_pipeline.ipynb`.
2. **`POST /predict/`** — envía créditos de `data/df_test_small.csv` como JSON y recibe `p_0`, `p`, `p_1` por crédito.

**Pre-requisito**: el servidor debe estar corriendo. Desde `../model_serving/`:

```bash
uv run uvicorn main:app --reload
```

## 1 — Subida de los pickles vía `/upload_model/`

Mandamos los tres artefactos como `multipart/form-data`. Los nombres de campo (`preprocessing`, `filtering`, `model`) deben coincidir exactamente con los parámetros del endpoint.

In [11]:
import requests

API_URL = "http://127.0.0.1:8000"

with open("model_serving/models/preprocessor.pkl", "rb") as f_pre, \
     open("model_serving/models/filter.pkl", "rb") as f_filt, \
     open("model_serving/models/model.pkl", "rb") as f_model:
    files = {
        "preprocessing": ("preprocessing.pkl", f_pre, "application/octet-stream"),
        "filtering":     ("filtering.pkl",     f_filt, "application/octet-stream"),
        "model":         ("model.pkl",         f_model, "application/octet-stream"),
    }
    resp = requests.post(f"{API_URL}/model/upload", files=files)

resp.raise_for_status()
resp.json()

{'status': 'ok',
 'saved': {'preprocessing': {'path': 'C:\\Users\\lalda\\Documents\\CUNEF\\IV\\datos\\Practica2\\model_serving\\models\\preprocessing.pkl',
   'size': 64414120},
  'filtering': {'path': 'C:\\Users\\lalda\\Documents\\CUNEF\\IV\\datos\\Practica2\\model_serving\\models\\filtering.pkl',
   'size': 6585419},
  'model': {'path': 'C:\\Users\\lalda\\Documents\\CUNEF\\IV\\datos\\Practica2\\model_serving\\models\\model.pkl',
   'size': 43949002}},
 'timestamp': '2026-05-15T16:41:08.769747'}

## 2 — Predicción de ejemplos de test vía `/predict/`

Leemos `data/df_test_small.csv` y mandamos **5 créditos** como una lista JSON. El endpoint acepta tanto `dict` (una fila) como `list[dict]` (batch).

> **Por qué un batch y no una fila suelta:** el preprocesador imputa nulos numéricos con la **mediana del lote actual** (no la del entrenamiento). Con una sola fila, si esa fila tiene un `NaN`, `median()` también es `NaN`, no rellena nada y `PolynomialFeatures` aborta. Con varias filas la mediana existe y la imputación funciona.
>
> Además, `requests` con `json=` usa `allow_nan=False`, así que serializamos primero con `pandas.to_json` (mapea `NaN → null`).

In [ ]:
#ME SALTA ERROR PERO EL MAIN ESTA ADPATDO A LO QUE PIDE LA PRÁCTICA.  

import json
import pandas as pd

df_test = pd.read_csv("data/df_test_small.csv")
muestra = df_test[df_test["desc"].notna()].head(5)

# pandas.to_json convierte NaN -> null (JSON valido); evita el error
# `Out of range float values are not JSON compliant` de requests.
ejemplos = json.loads(muestra.to_json(orient="records"))

resp = requests.post(f"{API_URL}/predict/", json=ejemplos)
resp.raise_for_status()
pd.DataFrame(resp.json())

STATUS: 400
TEXT: {"detail":"argument of type 'method' is not iterable"}


HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8000/predict/